In [55]:
import numpy as np
import xarray as xr
import scipy.io as sio
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, interp2d
from matplotlib import cm,colors
import pickle
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import pandas as pd


In [56]:
melange_area_path = '/home/m484s199/iceberg_py/dev/geoms/helheim/melange_area/'
mel_list = sorted( [file for file in os.listdir(melange_area_path) if file.endswith('gpkg') ] )
date_list = [gpd.pd.to_datetime(file[8:18]) for file in mel_list]
gdf_list = [gpd.read_file(f'{melange_area_path}{file}') for file in mel_list]
    


In [57]:
mel_list

['helhiem_2016-04-24_area.gpkg',
 'helhiem_2018-05-04_area.gpkg',
 'helhiem_2020-09-03_area.gpkg',
 'helhiem_2023-07-27_area.gpkg',
 'helhiem_2024-08-22_area.gpkg']

In [58]:
date_list

[Timestamp('2016-04-24 00:00:00'),
 Timestamp('2018-05-04 00:00:00'),
 Timestamp('2020-09-03 00:00:00'),
 Timestamp('2023-07-27 00:00:00'),
 Timestamp('2024-08-22 00:00:00')]

In [59]:
gdf_list[0].area[0]

np.float64(132528531.77252682)

In [60]:
area_dict = {date:gdf.area[0] for date, gdf in zip(date_list, gdf_list)}
area_dict

{Timestamp('2016-04-24 00:00:00'): np.float64(132528531.77252682),
 Timestamp('2018-05-04 00:00:00'): np.float64(142934146.47607896),
 Timestamp('2020-09-03 00:00:00'): np.float64(141325864.68025035),
 Timestamp('2023-07-27 00:00:00'): np.float64(142934146.47607896),
 Timestamp('2024-08-22 00:00:00'): np.float64(134061538.63418496)}

In [61]:
temp_dict = {'min':4.0,
            'avg': 5.4,
            'max': 6.3} #from CTD AW average TF

urel_dict = {'min':0.07,
            'avg': 0.13,
            'max': 0.20} #model vel runs

In [62]:
run_type = 'max'

dt = 50
psw = 1024 #kg m3
csw = 3974 #J kg-1 C-1
day2sec = 86400
depth = 450
temp = temp_dict[run_type]
coeff_1_path = f'/home/m484s199/iceberg_py/data/iceberg_model_output_melt_fix/helheim/{run_type}/'


coeff_1_list = sorted([nc for nc in os.listdir(coeff_1_path) if nc.endswith('nc')])

def Qaw_calc(area, dt = dt, psw = psw, csw = csw, depth = depth, temp = temp):
    
    vol = area * depth
    
    Qaw = psw * csw * ( (vol * temp) / (dt * day2sec) )

    return Qaw

dQ_dt_HEL_CTD_avg_dict = {date:Qaw_calc(vol) for date, vol in area_dict.items()}

# dQ_dt_HEL_CTD_avg = psw * csw * ( (Volume_test * 5.4) / (dt * day2sec) )

os.chdir(coeff_1_path)

cols = ['coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'melt_rate_mday', 'percentage']

In [63]:
dQ_dt_HEL_CTD_avg_dict

{Timestamp('2016-04-24 00:00:00'): np.float64(353921154897.4225),
 Timestamp('2018-05-04 00:00:00'): np.float64(381709640320.47015),
 Timestamp('2020-09-03 00:00:00'): np.float64(377414678752.8196),
 Timestamp('2023-07-27 00:00:00'): np.float64(381709640320.47015),
 Timestamp('2024-08-22 00:00:00'): np.float64(358015092645.6727)}

In [64]:
# dQ_dt_HEL_CTD_avg/1e9

In [65]:
coeff_1_list = sorted([nc for nc in os.listdir(coeff_1_path) if nc.endswith('nc')])

cols = ['date', 'coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'percentage']
coef_1_dict = {}

series_list1 = []
for i,nc in enumerate(coeff_1_list):
    
    Qib = xr.open_dataset(f'{coeff_1_path}{nc}')
    Qib_val = Qib.Qib.data
    date = nc[:10]
    date = pd.to_datetime(date)
    TF = nc.split('_')[6]
    urel_val = nc.split('_')[-1].split('.')[1]
    
    # print(f'coeff 1: {percentage:.2f}')
    print(f'{date}')
    coef_1_dict['date'] = date
    coef_1_dict['coeff'] = 1
    coef_1_dict['urel'] = urel_dict[run_type]
    coef_1_dict['tf'] = temp_dict[run_type]
    coef_1_dict['dt'] = dt
    coef_1_dict['Qib'] = (Qib_val/1e11)
    # coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg #(dQ_dt_dict[TF])
    coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg_dict[date]
    
    coef_1_dict['melt_rate_avg_m3s'] = Qib.melt_rate_integrated.data
    
    coef_1_dict['percentage'] = f'{(Qib_val/dQ_dt_HEL_CTD_avg_dict[date])*100:.2f}' #f'{(Qib_val/dQ_dt_dict[TF])*100:.2f}'
    
    series = pd.Series(coef_1_dict)
    series_list1.append(series)

df_50 = pd.DataFrame(series_list1, columns=cols)   



2016-04-24 00:00:00
2018-05-04 00:00:00
2020-09-03 00:00:00
2023-07-27 00:00:00
2024-08-22 00:00:00


In [66]:
!pwd

/home/m484s199/iceberg_py/data/iceberg_model_output_melt_fix/helheim/max


In [67]:
nc.split('_')

['2024-08-22', 'helheim', 'coeff', '1', 'CTD', 'constant', 'UREL', '2.nc']

In [68]:
df_50

,date,coeff,urel,tf,dt,Qib,Qaww,melt_rate_avg_m3s,percentage
0,2016-04-24,1,0.2,6.3,50,0.253862,3.539212e+11,75.77978893647814,7.17
1,2018-05-04,1,0.2,6.3,50,0.267541,3.817096e+11,79.86286130467931,7.01
2,2020-09-03,1,0.2,6.3,50,0.312470,3.774147e+11,93.27475988526368,8.28
3,2023-07-27,1,0.2,6.3,50,0.572805,3.817096e+11,170.98655616713017,15.01
4,2024-08-22,1,0.2,6.3,50,0.441935,3.580151e+11,131.9209797169548,12.34


In [69]:
df_50['percentage'].astype(np.float64).mean()

np.float64(9.962)

In [70]:
run_type = 'max'

dt = 150
psw = 1024 #kg m3
csw = 3974 #J kg-1 C-1
day2sec = 86400
depth = 450
temp = temp_dict[run_type]

def Qaw_calc(area, dt = dt, psw = psw, csw = csw, depth = depth, temp = temp):
    
    vol = area * depth
    
    Qaw = psw * csw * ( (vol * temp) / (dt * day2sec) )

    return Qaw

dQ_dt_HEL_CTD_avg_dict = {date:Qaw_calc(vol) for date, vol in area_dict.items()}

# dQ_dt_HEL_CTD_avg = psw * csw * ( (Volume_test * 5.4) / (dt * day2sec) )

os.chdir(coeff_1_path)

cols = ['coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'melt_rate_mday', 'percentage']

In [71]:
cols = ['date', 'coeff', 'urel', 'tf', 'dt', 'Qib', 'Qaww', 'melt_rate_avg_m3s', 'percentage']
coef_1_dict = {}

series_list1 = []
for i,nc in enumerate(coeff_1_list):
    
    Qib = xr.open_dataset(f'{coeff_1_path}{nc}')
    Qib_val = Qib.Qib.data
    date = nc[:10]
    date = pd.to_datetime(date)
    TF = nc.split('_')[6]
    urel_val = nc.split('_')[-1].split('.')[1]
    
    # print(f'coeff 1: {percentage:.2f}')
    print(f'{date}')
    coef_1_dict['date'] = date
    coef_1_dict['coeff'] = 1
    coef_1_dict['urel'] = urel_dict[run_type]
    coef_1_dict['tf'] = temp_dict[run_type]
    coef_1_dict['dt'] = dt
    coef_1_dict['Qib'] = (Qib_val/1e11)
    # coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg #(dQ_dt_dict[TF])
    coef_1_dict['Qaww'] = dQ_dt_HEL_CTD_avg_dict[date]
    
    coef_1_dict['melt_rate_avg_m3s'] = Qib.melt_rate_integrated.data
    
    coef_1_dict['percentage'] = f'{(Qib_val/dQ_dt_HEL_CTD_avg_dict[date])*100:.2f}' #f'{(Qib_val/dQ_dt_dict[TF])*100:.2f}'
    
    series = pd.Series(coef_1_dict)
    series_list1.append(series)

df_150 = pd.DataFrame(series_list1, columns=cols)   

2016-04-24 00:00:00
2018-05-04 00:00:00
2020-09-03 00:00:00
2023-07-27 00:00:00
2024-08-22 00:00:00


In [72]:
df_150

,date,coeff,urel,tf,dt,Qib,Qaww,melt_rate_avg_m3s,percentage
0,2016-04-24,1,0.2,6.3,150,0.253862,1.179737e+11,75.77978893647814,21.52
1,2018-05-04,1,0.2,6.3,150,0.267541,1.272365e+11,79.86286130467931,21.03
2,2020-09-03,1,0.2,6.3,150,0.312470,1.258049e+11,93.27475988526368,24.84
3,2023-07-27,1,0.2,6.3,150,0.572805,1.272365e+11,170.98655616713017,45.02
4,2024-08-22,1,0.2,6.3,150,0.441935,1.193384e+11,131.9209797169548,37.03


In [73]:
df_150['percentage'].astype(np.float64).mean()

np.float64(29.887999999999998)